[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/066.%20The%20Workforce%20Scheduling%20for%20Peak_Off-Peak%20Problem/P66-Tier-3_executed.ipynb)

---


# 66. The Workforce Scheduling for Peak/Off-Peak Problem
## Tier 3: Grey Wolf Optimizer

### Purpose and workflow

This notebook uses a Grey Wolf Optimizer (GWO), a population-based metaheuristic, to search for a low-cost workforce schedule. The input is a list of employees, a set of time periods, the staffing demand in each period, and labor and staffing-imbalance costs. The output is the best schedule found during the search, together with its cost, convergence history, and population-diversity history.

A metaheuristic is a general search strategy: it can explore a large set of candidate schedules, but it does not prove that the best schedule it finds is globally optimal. The three leading candidates are called alpha, beta, and delta; the remaining candidates are omega wolves. Each omega candidate is moved using information from all three leaders.

### Modeling assumptions and scope

- Each employee-period assignment represents one labor hour. An employee may work at most `max_hours` periods in the schedule.
- A period is represented internally by its zero-based position in `time_periods`; the demand entry at the same position applies to it. The example uses consecutive period labels.
- Employees are available in every period. The model does not include shift lengths, breaks, rest rules, skills, employee preferences, or period-to-period availability.
- Labor cost is the employee's hourly rate multiplied by assigned periods. Understaffing and overstaffing incur their specified per-person-per-period penalties.
- The continuous GWO position is converted to a binary schedule by thresholding at 0.5. If thresholding assigns an employee to more than `max_hours` periods, only that employee's highest-position assignments are retained.
- The fitness function adds an extra penalty for unmet demand so that schedules with shortages are strongly discouraged. This search penalty is separate from the understaffing cost used to report schedule cost.

### Mathematical model and search representation

Let $E$ be the set of employees and $T$ the set of time periods. The binary decision variable $x_{et}$ is 1 when employee $e$ works in period $t$, and 0 otherwise. Let $r_e$ be employee $e$'s hourly rate, $d_t$ the required staffing in period $t$, $c_u$ the understaffing penalty, and $c_o$ the overstaffing penalty. The modeled schedule cost is

$$
C(x) = \sum_{e \in E} r_e \sum_{t \in T} x_{et}
+ c_u \sum_{t \in T} \max\!\left(0, d_t - \sum_{e \in E} x_{et}\right)
+ c_o \sum_{t \in T} \max\!\left(0, \sum_{e \in E} x_{et} - d_t\right).
$$

The schedule representation enforces the employee-hours limit

$$
\sum_{t \in T} x_{et} \leq h_e \qquad (e \in E),
$$

where $h_e$ is employee $e$'s `max_hours`. Demand is encouraged through the cost and the additional search penalty; it is not a hard constraint. Specifically, the implementation ranks a schedule using fitness

$$
F(x) = \frac{1}{1 + C(x) + 1000\displaystyle\sum_{t \in T}\max\!\left(0, d_t - \sum_{e \in E} x_{et}\right)}.
$$

Thus, larger fitness means a more desirable candidate. The additional shortage penalty affects candidate ranking, while `calculate_schedule_cost` reports $C(x)$ alone. The constant 1000 is a search penalty, not a labor-cost parameter.

A wolf's position is a real-valued vector with one coordinate for every employee-period pair, so its dimension is $D = |E||T|$. To move an omega wolf, the optimizer calculates candidate coordinates relative to each leader and averages the three results. For leader $L \in \{\alpha,\beta,\delta\}$ and coordinate $j$, the update is

$$
D_L^j = \left|C_L^j X_L^j - X^j\right|, \qquad
X_L^j = X_L^j - A_L^j D_L^j,
$$

where $X^j$ is the current wolf coordinate, $X_L^j$ is the leader's coordinate, and

$$
A_L^j = 2a r_{1,L}^j - a, \qquad C_L^j = 2r_{2,L}^j.
$$

The random values $r_{1,L}^j$ and $r_{2,L}^j$ are drawn uniformly from $[0,1]$. The control parameter $a$ decreases linearly from its initial value toward its final value; it controls the magnitude of position updates. The new coordinate is the mean of the three leader-based coordinates, clipped to $[0,1]$. The vector is then decoded to a binary schedule. In the code, these operations appear in `hunting_behavior`, with the coordinate arrays stored as each wolf's `position`.

The base state of the search is the randomly initialized pack. Each iteration updates omega positions, evaluates their decoded schedules, reorders the pack by fitness, and records convergence and diversity. With pack size $P$, iteration count $I$, and position dimension $D$, position updates require $O(IPD)$ coordinate work; sorting adds $O(IP\log P)$. Schedule evaluation also depends on the number of employees and periods. Because the search is stochastic and uses a finite pack and iteration limit, convergence is evidence about the run, not a proof of global optimality.

### What to examine

- **Leader costs:** alpha, beta, and delta are the three highest-fitness candidates, not necessarily the three lowest-cost schedules when shortage penalties affect fitness ranking.
- **Convergence:** a stable alpha cost means the leading schedule's reported cost has not changed during that part of the run.
- **Diversity:** the average Euclidean distance of wolf positions from the pack's mean position. It is measured in the encoded coordinate space and depends on the number of employee-period coordinates.
- **Coverage:** compare assigned staff with demand in every period; a low reported cost does not by itself establish that every demand requirement is met.
- **Sensitivity results:** different pack sizes and iteration limits can produce different outcomes. Their measured run times depend on the execution environment.


In [ ]:
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import time

plt.style.use("default")
sns.set_palette("husl")
random.seed(42)

In [ ]:
class GreyWolfOptimizer:
    """Apply Grey Wolf Optimizer search to workforce schedules."""

    def __init__(
        self,
        employees: List[Dict],
        time_periods: List[int],
        demand: List[int],
        cost_params: Dict,
        pack_size: int = 20,
        max_iterations: int = 100,
    ):
        """Store the scheduling instance and GWO search settings."""
        if len(time_periods) != len(demand):
            raise ValueError("Each time period must have one demand value.")
        if len({employee["id"] for employee in employees}) != len(employees):
            raise ValueError("Employee IDs must be unique.")
        if pack_size < 3:
            raise ValueError("The pack must contain at least three wolves.")
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1.")
        if any(value < 0 for value in demand):
            raise ValueError("Demand values must be non-negative.")

        self.employees = employees
        self.time_periods = time_periods
        self.demand = demand
        self.cost_params = cost_params
        self.pack_size = pack_size
        self.max_iterations = max_iterations

        self.a_initial = 2.0
        self.a_final = 0.0

        self.pack = []
        self.alpha = None
        self.beta = None
        self.delta = None
        self.convergence_history = []
        self.diversity_history = []

    def encode_schedule(self, schedule: Dict) -> np.ndarray:
        """Flatten a binary employee-by-period schedule into a position."""
        position = np.zeros((len(self.employees), len(self.time_periods)))

        for employee_index, employee in enumerate(self.employees):
            employee_id = employee["id"]
            for period_index in schedule.get(employee_id, []):
                if 0 <= period_index < len(self.time_periods):
                    position[employee_index, period_index] = 1.0

        return position.flatten()

    def decode_schedule(self, position: np.ndarray) -> Dict:
        """Convert a continuous position to a binary, hours-feasible schedule."""
        expected_size = len(self.employees) * len(self.time_periods)
        if position.size != expected_size:
            raise ValueError("Position length does not match the schedule dimensions.")

        position_matrix = position.reshape(
            (len(self.employees), len(self.time_periods))
        )
        schedule = {}

        for employee_index, employee in enumerate(self.employees):
            employee_id = employee["id"]
            eligible_periods = [
                period_index
                for period_index in range(len(self.time_periods))
                if position_matrix[employee_index, period_index] > 0.5
            ]
            max_hours = max(0, int(employee.get("max_hours", 8)))

            # Keeping the strongest coordinates enforces the employee-hours limit
            # while preserving a deterministic mapping from position to schedule.
            selected_periods = sorted(
                eligible_periods,
                key=lambda period_index: (
                    -position_matrix[employee_index, period_index],
                    period_index,
                ),
            )[:max_hours]
            schedule[employee_id] = sorted(selected_periods)

        return schedule

    def generate_random_schedule(self) -> Dict:
        """Build a random schedule without exceeding employee-hours limits."""
        schedule = {employee["id"]: [] for employee in self.employees}

        for period_index, required in enumerate(self.demand):
            available_employees = [
                employee
                for employee in self.employees
                if self.is_employee_available(employee, period_index, schedule)
            ]

            selected = random.sample(
                available_employees,
                min(required, len(available_employees)),
            )
            for employee in selected:
                schedule[employee["id"]].append(period_index)

        return schedule

    def is_employee_available(
        self, employee: Dict, period: int, schedule: Dict
    ) -> bool:
        """Return whether an employee can take another period assignment."""
        employee_id = employee["id"]
        current_assignments = len(schedule.get(employee_id, []))
        max_hours = max(0, int(employee.get("max_hours", 8)))
        return current_assignments < max_hours

    def calculate_fitness(self, schedule: Dict) -> float:
        """Return inverse penalized cost, so a higher value ranks better."""
        total_cost = self.calculate_schedule_cost(schedule)
        shortage_penalty = 0

        for period_index, required in enumerate(self.demand):
            assigned = sum(
                1
                for periods in schedule.values()
                if period_index in periods
            )
            if assigned < required:
                shortage_penalty += (required - assigned) * 1000

        return 1.0 / (1.0 + total_cost + shortage_penalty)

    def calculate_schedule_cost(self, schedule: Dict) -> float:
        """Calculate labor, understaffing, and overstaffing costs."""
        total_cost = 0.0

        for employee_id, periods in schedule.items():
            employee = next(
                employee
                for employee in self.employees
                if employee["id"] == employee_id
            )
            hourly_rate = employee.get(
                "hourly_rate", self.cost_params["default_labor_cost"]
            )
            total_cost += len(periods) * hourly_rate

        for period_index, required in enumerate(self.demand):
            assigned = sum(
                1
                for periods in schedule.values()
                if period_index in periods
            )

            if assigned < required:
                total_cost += (required - assigned) * self.cost_params[
                    "understaff_penalty"
                ]
            elif assigned > required:
                total_cost += (assigned - required) * self.cost_params[
                    "overstaff_penalty"
                ]

        return total_cost

    def initialize_pack(self):
        """Create and rank the initial population of random schedules."""
        self.pack = []

        for _ in range(self.pack_size):
            schedule = self.generate_random_schedule()
            position = self.encode_schedule(schedule)
            wolf = {
                "position": position,
                "schedule": schedule,
                "fitness": self.calculate_fitness(schedule),
                "cost": self.calculate_schedule_cost(schedule),
            }
            self.pack.append(wolf)

        self.update_hierarchy()

    def update_hierarchy(self):
        """Rank the pack and identify its three leading wolves."""
        self.pack.sort(key=lambda wolf: wolf["fitness"], reverse=True)
        self.alpha = self.pack[0]
        self.beta = self.pack[1]
        self.delta = self.pack[2]

    def calculate_diversity(self) -> float:
        """Measure mean Euclidean distance from the pack's mean position."""
        if len(self.pack) < 2:
            return 0.0

        positions = np.array([wolf["position"] for wolf in self.pack])
        mean_position = np.mean(positions, axis=0)
        distances = [
            np.linalg.norm(position - mean_position) for position in positions
        ]
        return float(np.mean(distances))

    def hunting_behavior(self, iteration: int):
        """Update omega positions using the alpha, beta, and delta positions."""
        a = self.a_initial - (self.a_initial - self.a_final) * (
            iteration / self.max_iterations
        )

        for wolf_index in range(3, len(self.pack)):
            wolf = self.pack[wolf_index]
            current_position = wolf["position"]
            new_position = np.zeros_like(current_position)

            for dimension in range(len(current_position)):
                leader_positions = []

                for leader in (self.alpha, self.beta, self.delta):
                    r1 = random.random()
                    r2 = random.random()
                    coefficient_a = 2 * a * r1 - a
                    coefficient_c = 2 * r2
                    distance = abs(
                        coefficient_c * leader["position"][dimension]
                        - current_position[dimension]
                    )
                    leader_positions.append(
                        leader["position"][dimension]
                        - coefficient_a * distance
                    )

                new_position[dimension] = np.clip(
                    np.mean(leader_positions), 0.0, 1.0
                )

            wolf["position"] = new_position
            wolf["schedule"] = self.decode_schedule(new_position)
            wolf["fitness"] = self.calculate_fitness(wolf["schedule"])
            wolf["cost"] = self.calculate_schedule_cost(wolf["schedule"])

    def optimize(self) -> Dict:
        """Run the GWO iterations and return the best schedule found."""
        print(
            f"Starting Grey Wolf Optimizer with {self.pack_size} wolves "
            f"for {self.max_iterations} iterations"
        )

        self.convergence_history = []
        self.diversity_history = []
        self.initialize_pack()
        initial_best_cost = self.alpha["cost"]
        print(f"Initial best cost: ${initial_best_cost:.2f}")

        self.convergence_history.append(
            {
                "iteration": 0,
                "alpha_cost": self.alpha["cost"],
                "beta_cost": self.beta["cost"],
                "delta_cost": self.delta["cost"],
                "avg_cost": np.mean([wolf["cost"] for wolf in self.pack]),
            }
        )
        self.diversity_history.append(self.calculate_diversity())

        for iteration in range(self.max_iterations):
            self.hunting_behavior(iteration)
            self.update_hierarchy()

            completed_iterations = iteration + 1
            self.convergence_history.append(
                {
                    "iteration": completed_iterations,
                    "alpha_cost": self.alpha["cost"],
                    "beta_cost": self.beta["cost"],
                    "delta_cost": self.delta["cost"],
                    "avg_cost": np.mean(
                        [wolf["cost"] for wolf in self.pack]
                    ),
                }
            )
            diversity = self.calculate_diversity()
            self.diversity_history.append(diversity)

            if completed_iterations % 10 == 0:
                print(
                    f"Iteration {completed_iterations}: "
                    f"Best cost = ${self.alpha['cost']:.2f}, "
                    f"Diversity = {diversity:.4f}"
                )

        print("\nOptimization completed.")
        print(f"Final best cost: ${self.alpha['cost']:.2f}")
        print(
            f"Total improvement: "
            f"${initial_best_cost - self.alpha['cost']:.2f}"
        )

        return {
            "best_schedule": self.alpha["schedule"],
            "best_cost": self.alpha["cost"],
            "convergence_history": self.convergence_history,
            "diversity_history": self.diversity_history,
            "final_pack": self.pack,
        }

In [ ]:
employees = [
    {"id": 1, "name": "Alice", "hourly_rate": 22, "max_hours": 8},
    {"id": 2, "name": "Bob", "hourly_rate": 20, "max_hours": 8},
    {"id": 3, "name": "Charlie", "hourly_rate": 24, "max_hours": 6},
    {"id": 4, "name": "Diana", "hourly_rate": 21, "max_hours": 8},
    {"id": 5, "name": "Eve", "hourly_rate": 19, "max_hours": 8},
    {"id": 6, "name": "Frank", "hourly_rate": 23, "max_hours": 8},
]

time_periods = [0, 1, 2, 3, 4, 5, 6, 7]
demand = [1, 2, 4, 3, 2, 3, 2, 1]

cost_params = {
    "default_labor_cost": 20,
    "understaff_penalty": 50,
    "overstaff_penalty": 10,
}

print("Grey Wolf Optimizer test data:")
print(f"Employees: {len(employees)}")
print(f"Time periods: {len(time_periods)}")
print(f"Demand pattern: {demand}")
print(f"Total demand: {sum(demand)} staff-periods")

In [ ]:
gwo = GreyWolfOptimizer(
    employees=employees,
    time_periods=time_periods,
    demand=demand,
    cost_params=cost_params,
    pack_size=20,
    max_iterations=100,
)

start_time = time.time()
results = gwo.optimize()
optimization_time = time.time() - start_time

print(f"\nOptimization time: {optimization_time:.2f} seconds")
print(f"Final best cost: ${results['best_cost']:.2f}")

In [ ]:
best_schedule = results["best_schedule"]
employee_by_id = {employee["id"]: employee for employee in employees}

schedule_summary = []
for employee_id, period_indices in best_schedule.items():
    employee = employee_by_id[employee_id]
    if period_indices:
        schedule_summary.append(
            {
                "employee": employee["name"],
                "periods": [period_index + 1 for period_index in period_indices],
                "periods_worked": len(period_indices),
                "labor_cost": len(period_indices) * employee.get(
                    "hourly_rate", cost_params["default_labor_cost"]
                ),
            }
        )

schedule_df = pd.DataFrame(
    schedule_summary,
    columns=["employee", "periods", "periods_worked", "labor_cost"],
)

print("Schedule returned by the alpha wolf:")
print(schedule_df.to_string(index=False))

print(f"\nTotal labor cost: ${schedule_df['labor_cost'].sum():.2f}")

total_understaff = 0
total_overstaff = 0
print("\nDemand satisfaction:")

for period_index, required in enumerate(demand):
    assigned = sum(
        1
        for period_indices in best_schedule.values()
        if period_index in period_indices
    )
    understaff = max(0, required - assigned)
    overstaff = max(0, assigned - required)
    total_understaff += understaff
    total_overstaff += overstaff
    status = "Met" if assigned >= required else "Not met"

    print(
        f"Period {period_index + 1}: Required {required}, "
        f"Assigned {assigned} ({status})"
    )

for employee in employees:
    assigned_periods = best_schedule[employee["id"]]
    assert len(assigned_periods) <= employee.get("max_hours", 8)

assert np.isclose(
    gwo.calculate_schedule_cost(best_schedule), results["best_cost"]
)

print(f"\nTotal understaffed staff-periods: {total_understaff}")
print(f"Total overstaffed staff-periods: {total_overstaff}")
print("Employee-hours limits and reported schedule cost have been checked.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Grey Wolf Optimizer Analysis", fontsize=16, fontweight="bold")

history = results["convergence_history"]
iterations = [entry["iteration"] for entry in history]

ax1 = axes[0, 0]
ax1.plot(
    iterations,
    [entry["alpha_cost"] for entry in history],
    "r-",
    linewidth=2,
    label="Alpha (highest fitness)",
)
ax1.plot(
    iterations,
    [entry["beta_cost"] for entry in history],
    "b-",
    linewidth=1.5,
    label="Beta (second-highest fitness)",
)
ax1.plot(
    iterations,
    [entry["delta_cost"] for entry in history],
    "g-",
    linewidth=1.5,
    label="Delta (third-highest fitness)",
)
ax1.plot(
    iterations,
    [entry["avg_cost"] for entry in history],
    "k--",
    linewidth=1,
    alpha=0.7,
    label="Pack average",
)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Reported schedule cost ($)")
ax1.set_title("Leader and Pack Costs")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(iterations, results["diversity_history"], color="purple", linewidth=2)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Mean Euclidean distance from pack mean")
ax2.set_title("Population Diversity")
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
period_indices = list(range(len(demand)))
coverage = [
    sum(
        1
        for assigned_periods in best_schedule.values()
        if period_index in assigned_periods
    )
    for period_index in period_indices
]
display_periods = [period_index + 1 for period_index in period_indices]
ax3.plot(
    display_periods,
    demand,
    "ro-",
    label="Demand",
    linewidth=2,
    markersize=8,
)
ax3.plot(
    display_periods,
    coverage,
    "bs-",
    label="Staff assigned",
    linewidth=2,
    markersize=8,
)
ax3.set_xlabel("Time period")
ax3.set_ylabel("Number of staff")
ax3.set_title("Demand and Staff Coverage")
ax3.set_xticks(display_periods)
ax3.legend()
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
final_pack = results["final_pack"]
final_costs = [wolf["cost"] for wolf in final_pack]
pack_colors = [
    "red" if rank == 0 else
    "blue" if rank == 1 else
    "green" if rank == 2 else
    "gray"
    for rank in range(len(final_pack))
]
ax4.scatter(
    range(len(final_pack)),
    final_costs,
    c=pack_colors,
    s=100,
    alpha=0.7,
)
ax4.set_xlabel("Wolf rank (ordered by fitness)")
ax4.set_ylabel("Reported schedule cost ($)")
ax4.set_title("Final Pack Cost Distribution")
ax4.grid(True, alpha=0.3)
for color, label in [
    ("red", "Alpha"),
    ("blue", "Beta"),
    ("green", "Delta"),
    ("gray", "Omega"),
]:
    ax4.scatter([], [], c=color, s=100, label=label)
ax4.legend(title="Pack role")

plt.tight_layout()
plt.show()

In [ ]:
print("Demonstrating Grey Wolf hunting behavior:")
print("=" * 48)

gwo_demo = GreyWolfOptimizer(
    employees=employees[:4],
    time_periods=time_periods[:6],
    demand=demand[:6],
    cost_params=cost_params,
    pack_size=8,
    max_iterations=5,
)
gwo_demo.initialize_pack()

print(f"Initial alpha cost: ${gwo_demo.alpha['cost']:.2f}")
print(f"Initial beta cost: ${gwo_demo.beta['cost']:.2f}")
print(f"Initial delta cost: ${gwo_demo.delta['cost']:.2f}")
print(f"Initial pack diversity: {gwo_demo.calculate_diversity():.4f}")

print("\nIteration 1: one hunting update")
omega = gwo_demo.pack[3]
omega_position_before = omega["position"].copy()
omega_cost_before = omega["cost"]
leader_positions = {
    "Alpha": gwo_demo.alpha["position"].copy(),
    "Beta": gwo_demo.beta["position"].copy(),
    "Delta": gwo_demo.delta["position"].copy(),
}

gwo_demo.hunting_behavior(0)
gwo_demo.update_hierarchy()

position_change = np.linalg.norm(
    omega["position"] - omega_position_before
)
print(f"Omega position change: {position_change:.4f}")
print(f"Omega cost before update: ${omega_cost_before:.2f}")
print(f"Omega cost after update: ${omega['cost']:.2f}")

for leader_name, leader_position in leader_positions.items():
    distance = np.linalg.norm(omega["position"] - leader_position)
    print(f"Distance from updated omega position to {leader_name}: {distance:.4f}")

print(f"\nAlpha cost after hunting: ${gwo_demo.alpha['cost']:.2f}")
print(f"Pack diversity after hunting: {gwo_demo.calculate_diversity():.4f}")

In [ ]:
parameter_tests = [
    {"pack_size": 10, "max_iterations": 50, "name": "Small pack"},
    {"pack_size": 20, "max_iterations": 100, "name": "Medium pack"},
    {"pack_size": 30, "max_iterations": 150, "name": "Large pack"},
]

results_comparison = []

for test_params in parameter_tests:
    print(f"\nTesting {test_params['name']}:")
    gwo_test = GreyWolfOptimizer(
        employees=employees,
        time_periods=time_periods,
        demand=demand,
        cost_params=cost_params,
        pack_size=test_params["pack_size"],
        max_iterations=test_params["max_iterations"],
    )

    start_time = time.time()
    test_result = gwo_test.optimize()
    elapsed_time = time.time() - start_time
    initial_cost = test_result["convergence_history"][0]["alpha_cost"]

    comparison_row = {
        "name": test_params["name"],
        "pack_size": test_params["pack_size"],
        "final_cost": test_result["best_cost"],
        "iterations": test_params["max_iterations"],
        "time_seconds": elapsed_time,
        "initial_cost": initial_cost,
        "improvement": initial_cost - test_result["best_cost"],
    }
    results_comparison.append(comparison_row)

    print(f"  Final cost: ${test_result['best_cost']:.2f}")
    print(f"  Time: {elapsed_time:.2f} seconds")
    print(f"  Improvement: ${comparison_row['improvement']:.2f}")

comparison_df = pd.DataFrame(results_comparison)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("GWO Parameter Sensitivity Analysis", fontsize=14, fontweight="bold")

bar_colors = ["#2ecc71", "#3498db", "#f39c12"]
bars = axes[0].bar(
    comparison_df["name"],
    comparison_df["final_cost"],
    color=bar_colors,
    alpha=0.8,
)
axes[0].set_ylabel("Final reported cost ($)")
axes[0].set_title("Solution Cost")
axes[0].grid(True, alpha=0.3, axis="y")
for bar, cost in zip(bars, comparison_df["final_cost"]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"${cost:.0f}",
        ha="center",
        va="bottom",
    )

bars = axes[1].bar(
    comparison_df["name"],
    comparison_df["time_seconds"],
    color=["#e74c3c", "#9b59b6", "#34495e"],
    alpha=0.8,
)
axes[1].set_ylabel("Elapsed time (seconds)")
axes[1].set_title("Computational Time")
axes[1].grid(True, alpha=0.3, axis="y")
for bar, elapsed_time in zip(bars, comparison_df["time_seconds"]):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{elapsed_time:.1f} s",
        ha="center",
        va="bottom",
    )

bars = axes[2].bar(
    comparison_df["name"],
    comparison_df["improvement"],
    color=["#16a085", "#27ae60", "#2980b9"],
    alpha=0.8,
)
axes[2].set_ylabel("Change in reported cost ($)")
axes[2].set_title("Initial-to-Final Cost Change")
axes[2].grid(True, alpha=0.3, axis="y")
for bar, improvement in zip(bars, comparison_df["improvement"]):
    axes[2].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"${improvement:.0f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

print("\nParameter analysis summary:")
print(
    comparison_df[
        ["name", "pack_size", "iterations", "final_cost", "time_seconds", "improvement"]
    ].to_string(index=False)
)

## Interpreting the results and choosing this tier

The GWO uses a population of schedules rather than following one schedule through a sequence of local changes. Its alpha, beta, and delta positions guide the movement of the other wolves, while the control parameter $a$ changes the size and character of those movements over the run. The method is useful as a search strategy when a schedule space is large or difficult to explore exhaustively, but it offers no certificate that the returned schedule is globally optimal.

Interpret the alpha, beta, and delta labels through fitness: leaders are ranked by the penalized fitness function, not by reported cost alone. Always inspect demand coverage and employee-hours limits alongside cost. A low-diversity population indicates that encoded positions have become more similar; it does not prove that the best possible schedule has been found.

The sensitivity experiment compares pack sizes and iteration limits while leaving the scheduling data and cost parameters unchanged. The random-number stream continues across runs, so results are reproducible in notebook order under the fixed seed, but each configuration does not receive an identical initial random population. Costs and elapsed times should therefore be interpreted as outcomes of these particular stochastic runs, not as universal performance guarantees.

This tier is appropriate when a flexible population-based search is useful and approximate solutions are acceptable. The schedule model remains limited to the assumptions stated above; additional operational rules would require explicit modeling before the schedules could be treated as operational plans.